# 01 — Compute Gaze Angle

**Purpose:** Combine IMU `pitch` with gaze `elevation` to produce a `gaze angle` time series.

## Pipeline

1. Set `DATA_DIR` to the folder containing your Pupil Neon export.
2. Load and synchronize IMU and gaze data (gaze is downsampled to the IMU frame rate).
3. Compute `gaze angle = pitch + elevation`.
4. Visualize the result and optionally save to `gaze_angle.csv`.

## Output

- `gaze_angle.csv` — columns: `timestamp [ns]`, `gaze angle [deg]`, `pitch [deg]`, `elevation [deg]`, `time_sec`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from neon_gaze.io import load_imu, load_gaze_positions, save_gaze_angle
from neon_gaze.processing import synchronize_gaze_to_imu, compute_gaze_angle
from neon_gaze.plotting import plot_gaze_angle, plot_gaze_histogram, plot_imu_yaw

## Configuration

Set `DEMO = True` to run with the small sample data shipped with this repo.  
Set `DEMO = False` (the default) to use your own Pupil Neon export.


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# If you are using this repo for the first time, set DEMO = True to
# run the pipeline on the included sample data.
DEMO = False

if DEMO:
    DATA_DIR = "../demo/input"
    OUTPUT_DIR = "../demo/output"
    CONDITION_LABEL = "demo"
else:
    DATA_DIR = "../data/session_01"
    OUTPUT_DIR = "../data/output"
    CONDITION_LABEL = "session_01"

SAVE_OUTPUT = False


## Step 1 — Load raw data

In [ ]:
imu_df = load_imu(os.path.join(DATA_DIR, "imu.csv"))
gaze_df = load_gaze_positions(os.path.join(DATA_DIR, "gaze_positions.csv"))

print("IMU data:")
display(imu_df.head())
print("\nGaze positions data:")
display(gaze_df.head())

## Step 2 — Synchronize and compute gaze angle

In [ ]:
merged_df = synchronize_gaze_to_imu(imu_df, gaze_df)
gaze_angle_df = compute_gaze_angle(merged_df)

print(f"Gaze angle dataframe: {len(gaze_angle_df)} rows")
display(gaze_angle_df.head())

## Step 3 — Visualize

In [ ]:
fig = plot_imu_yaw(imu_df, title=f"IMU Yaw Over Time — {CONDITION_LABEL}")
fig.show()

In [ ]:
fig = plot_gaze_angle(gaze_angle_df, title=f"Gaze Angle Over Time — {CONDITION_LABEL}")
fig.show()

In [ ]:
fig = plot_gaze_histogram(gaze_angle_df, title=f"Histogram of Gaze Angle — {CONDITION_LABEL}")
fig.show()

## Interactive metric explorer

Use the dropdown to toggle between `elevation` and `pitch` over time.

In [ ]:
from ipywidgets import widgets
import plotly.graph_objects as go

def plot_gaze_metric(metric):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=gaze_angle_df["time_sec"],
        y=gaze_angle_df[metric],
        mode="lines+markers",
        name=metric,
        line=dict(width=2),
        marker=dict(size=4),
    ))
    fig.update_layout(
        title=f"{metric} Over Time",
        xaxis_title="Time (sec)",
        yaxis_title=metric,
        yaxis=dict(range=[-90, 45]),
    )
    fig.show()

dropdown = widgets.Dropdown(
    options=["elevation [deg]", "pitch [deg]"],
    value="elevation [deg]",
    description="Metric:",
)
widgets.interact(plot_gaze_metric, metric=dropdown)

## Step 4 — Save output

In [ ]:
if SAVE_OUTPUT:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    save_gaze_angle(gaze_angle_df, os.path.join(OUTPUT_DIR, "gaze_angle.csv"))
else:
    print("SAVE_OUTPUT is False — set to True in the config cell to save.")